
# 02 — Risk-aware allocation

Forecasting `Gbps` is not provisioning. This notebook builds the decision layer the
project's title promises.

**The cost model.** For an allocation `A` against realised demand `y`:

$$C = c_{over}\,(A-y)^+ + \kappa\,c_{over}\,(y-A)^+$$

Under-provisioning costs κ times more than over-provisioning. The cost-minimising
allocation is then not the mean but the **τ-quantile** of the predictive
distribution, with

$$\tau^* = \frac{\kappa}{1+\kappa}$$

So the operator's cost ratio *selects the quantile*. κ = 10 → τ = 0.909.

In [ ]:
# --- Colab bootstrap -------------------------------------------------------
# Works in Colab and locally. In Colab, clone the repo first:
#     !git clone <repo-url> bwalloc && %cd bwalloc
import os, sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")
ROOT = Path.cwd()
while not (ROOT / "src" / "bwalloc").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

try:
    import xgboost  # noqa: F401
except ImportError:
    !pip install -q xgboost

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import bwalloc as bw
from bwalloc.plots import use_paper_style

bw.set_seed()
use_paper_style()
pd.set_option("display.width", 200)
RESULTS = ROOT / "experiments" / "results"
FIGURES = ROOT / "paper" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
print("bwalloc", bw.__version__, "| results:", RESULTS)

In [ ]:

from bwalloc.allocation import kappa_for_tau, optimal_tau

for kappa in (2, 5, 10, 20):
    print(f"kappa={kappa:2d}  ->  tau* = {optimal_tau(kappa):.4f}")
print()
# And the correspondence verified empirically rather than asserted:
rng = np.random.default_rng(0)
y = rng.gamma(shape=9, scale=10, size=200_000)
for kappa in (2, 10):
    grid = np.linspace(np.quantile(y, 0.4), np.quantile(y, 0.995), 400)
    costs = [np.mean(np.maximum(a - y, 0) + kappa * np.maximum(y - a, 0)) for a in grid]
    empirical = grid[int(np.argmin(costs))]
    print(f"kappa={kappa:2d}: cost-minimising allocation {empirical:7.2f}  "
          f"vs quantile at tau* {np.quantile(y, optimal_tau(kappa)):7.2f}")

## Conformal calibration

A quantile model that claims 95% rarely delivers 95%. Split conformal fixes that distribution-free — which matters precisely because there are only ~900 points and no parametric error model is credible.

In [ ]:

from bwalloc.context import assign_binary_groups
from bwalloc.data import load, sampling_profile
from bwalloc.features import FeatureConfig, build_features
from bwalloc.models import xgboost_point
from bwalloc.pipeline import coverage_table, run_allocation_backtest
from bwalloc.splits import rolling_origin

OPERATOR = "gp"
df = load(OPERATOR)
profile = sampling_profile(df)
X, y = build_features(df, profile, FeatureConfig())
groups = assign_binary_groups(df).reindex(X.index)
folds = rolling_origin(len(y), n_folds=8, calib_frac=0.30)

per_fold, allocations = run_allocation_backtest(
    X, y, folds, groups, model_factory=xgboost_point, taus=(0.80, 0.90, 0.95),
)
coverage_table(per_fold[per_fold["group"] == "ALL"], "coverage")


### Coverage is systematically short — and that is a finding

Achieved coverage lands *below* nominal almost everywhere, and the shortfall is
one-sided. That is the signature of exchangeability failing under temporal drift, not
of a bug: split conformal's finite-sample guarantee is conditional on exchangeability,
and a 55-day trace with a trend does not supply it.

`experiments/run_coverage_gate.py` quantifies it across every configuration.
`conformal.AdaptiveConformalInference` is the remedy — it updates the requested level
online from realised breaches, so long-run coverage converges without assuming
exchangeability at all. The `aci` row above is that method.

In [ ]:

gate = pd.read_csv(RESULTS / "coverage_gate.csv")
marginal = gate[(gate["group"] == "ALL") & (gate["operator"] == OPERATOR)]
marginal[["horizon_hours", "tau", "method", "n", "coverage", "gap", "passes", "decisive"]]

## The capacity–risk frontier

The headline comparison: the calibrated allocator against the fixed-margin rule operators actually use (`A = 1.3 × ŷ`), plus static peak allocation. Read it as *capacity required at an equal SLA violation rate*.

In [ ]:

from bwalloc.plots import plot_pareto

pareto = pd.read_csv(RESULTS / f"pareto_{OPERATOR}.csv")
fig, ax = plt.subplots(figsize=(6.5, 4.2))
plot_pareto(pareto, ax=ax)
ax.set_title(f"{OPERATOR.upper()} — capacity vs SLA risk")
fig.savefig(FIGURES / f"fig3_pareto_{OPERATOR}.png", dpi=200, bbox_inches="tight")

pd.read_csv(RESULTS / f"savings_{OPERATOR}.csv")


### Honest reading of the frontier

**The capacity-saving claim does not hold on these traces.** No conformal family
reaches a 1%, 2% or 5% violation target, and at the targets that are feasible the
saving against fixed-margin is negative or negligible.

The reason is structural and worth stating: the fixed-margin rule is *multiplicative*
(`A = 1.3 ŷ`) while additive conformal adds the same number of Gbps at 3 a.m. as at
peak. On a series whose level swings ~2× within a day that is a real handicap. The
`conformal_relative` family removes the confound by calibrating a percentage margin
instead — that is the like-for-like comparison, and it is the row to read.